In [9]:
pip install pandas sqlalchemy psycopg2-binary python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd

In [11]:
faturas_csv = ['DadosFaturas/Fatura_2025-03-20.csv', 'DadosFaturas/Fatura_2025-04-20.csv', 'DadosFaturas/Fatura_2025-05-20.csv', 
               'DadosFaturas/Fatura_2025-06-20.csv', 'DadosFaturas/Fatura_2025-07-20.csv', 'DadosFaturas/Fatura_2025-08-20.csv',
               'DadosFaturas/Fatura_2025-09-20.csv', 'DadosFaturas/Fatura_2025-10-20.csv', 'DadosFaturas/Fatura_2025-11-20.csv',
               'DadosFaturas/Fatura_2025-12-20.csv', 'DadosFaturas/Fatura_2026-01-20.csv', 'DadosFaturas/Fatura_2026-02-20.csv']

dados_faturas_list = []

for f in faturas_csv:
    fatura = pd.read_csv(f, sep=';')
    dados_faturas_list.append(fatura)

dados_faturas = pd.concat(dados_faturas_list)

In [12]:
dados_faturas = dados_faturas.drop(columns=["Descrição"])

dados_faturas.rename(columns={'Nome no Cartão': 'nome_cartao', 'Final do Cartão': 'final_cartao'}, inplace=True)
dados_faturas.rename(columns={'Data de Compra': 'data_completa'}, inplace=True)
dados_faturas.rename(columns={'Categoria': 'nome_categoria'}, inplace=True)
dados_faturas.rename(columns=
                    {'Valor (em R$)': 'valor_brl', 'Valor (em US$)': 'valor_usd', 'Parcela': 'parcela_texto', 
                     'Cotação (em R$)': 'cotacao_brl'
                    }, inplace=True)

In [13]:
print("Linhas carregadas:", len(dados_faturas))

Linhas carregadas: 1945


In [14]:
# DIM titular

titulares = dados_faturas[["nome_cartao", "final_cartao"]].drop_duplicates().copy()
titulares["id_titular"] = range(1, len(titulares) + 1)
dados_faturas = dados_faturas.merge(titulares, on=["nome_cartao", "final_cartao"])

In [15]:
print("titulares:", len(titulares))

titulares: 5


In [16]:
# DIM categoria

categorias = dados_faturas[["nome_categoria"]].drop_duplicates().copy()
categorias["id_categoria"] = range(1, len(categorias) + 1)
categorias.loc[categorias["nome_categoria"] == "-", "nome_categoria"] = "Não categorizado"
dados_faturas = dados_faturas.merge(categorias, on=["nome_categoria"])

In [17]:
print("Categorias:", len(categorias))

Categorias: 36


In [18]:
# DIM data_compra

from datetime import datetime

datas_compra = dados_faturas[["data_completa"]].drop_duplicates().copy()
datas_compra["id_data"] = range(1, len(datas_compra) + 1)

datas_compra["data_completa"] = pd.to_datetime(datas_compra["data_completa"], format="%d/%m/%Y")

datas_compra["dia"] = datas_compra["data_completa"].dt.day
datas_compra["mes"] = datas_compra["data_completa"].dt.month
datas_compra["ano"] = datas_compra["data_completa"].dt.year
datas_compra["trimestre"] = datas_compra["data_completa"].dt.quarter
datas_compra["dia_semana"] = datas_compra["data_completa"].dt.dayofweek

dados_faturas["data_completa"] = pd.to_datetime(dados_faturas["data_completa"], format="%d/%m/%Y")
dados_faturas = dados_faturas.merge(datas_compra, on=["data_completa"])

In [19]:
print("Datas:", len(datas_compra))

Datas: 343


In [20]:
# Tabela fato transacao

transacoes = dados_faturas[["id_titular", "id_data", "id_categoria", "valor_usd", "valor_brl", "cotacao_brl", "parcela_texto"]].copy()
transacoes["id_transacao"] = range(1, len(transacoes) + 1)

transacoes.loc[transacoes["parcela_texto"] == "Única", "parcela_texto"] = "1/1"
transacoes["num_parcela"] = transacoes["parcela_texto"].str.split("/").str[0]
transacoes["total_parcelas"] = transacoes["parcela_texto"].str.split("/").str[1]

In [21]:
print("transacoes:", len(transacoes))
print("duplicatas:", transacoes.duplicated().sum())

transacoes: 1903
duplicatas: 0


In [28]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port= os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

In [29]:
#Verifica se todos os IDs de titular nas transações existem na tabela titulares.
print(transacoes["id_titular"].isin(titulares["id_titular"]).all())

#Confere se todas as datas referenciadas existem na tabela datas.
print(transacoes["id_data"].isin(datas_compra["id_data"]).all())

#Confere se todas as categorias referenciadas existem na tabela categorias
print(transacoes["id_categoria"].isin(categorias["id_categoria"]).all())

#Se aparecer número > 0, significa que existem transações duplicadas.
print(transacoes.duplicated().sum())

#Saída esperada
#True
#True
#True
#0

True
True
True
0


In [31]:
print(len(titulares))
titulares.to_sql("titulares", engine, if_exists="append", index=False)

5


5

In [32]:
print(len(categorias))
categorias.to_sql("categorias", engine, if_exists="append", index=False)

36


36

In [33]:
print(len(datas_compra))
datas_compra.to_sql("datas_compra", engine, if_exists="append", index=False)

343


343

In [34]:
print(len(transacoes))
transacoes.to_sql("transacoes", engine, if_exists="append", index=False)

1903


903

In [35]:
from sqlalchemy import text

with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM titulares")).fetchone())
    print(conn.execute(text("SELECT COUNT(*) FROM categorias")).fetchone())
    print(conn.execute(text("SELECT COUNT(*) FROM datas_compra")).fetchone())
    print(conn.execute(text("SELECT COUNT(*) FROM transacoes")).fetchone())

(5,)
(36,)
(343,)
(1903,)
